# ENGR 418 - Facial Age Group Estimation
## Four-group CNN baseline: training and evaluation

This notebook trains a small convolutional neural network (CNN) for four combined age groups. It uses the same fixed train/validation/test row assignments as the eight-class baseline, with adjacent labels merged. Read explanations and run cells in order. **Training begins only when you run the training cell.**

The initial configuration uses 10 epochs and a modest model. Training time depends on your computer and available hardware.


## 1. What this baseline does

1. Load the dataset and the saved split table.
2. Convert each face image to a 200×200 RGB tensor with values from 0 to 1.
3. Train a CNN using weighted cross-entropy to account for the unequal class counts.
4. Select the checkpoint with the best validation macro-F1 score.
5. Evaluate the selected checkpoint once on the held-out test set.

Macro-F1 gives each age group equal weight, so the largest class cannot dominate the score. We will also report accuracy and a confusion matrix. The test set is reserved for the final evaluation; do not tune model settings based on its results.


In [ ]:
# If needed, install the dependencies in this notebook's environment:
# %pip install -U datasets pandas pillow matplotlib scikit-learn torch torchvision

from datasets import load_dataset
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

DATASET_ID = "prithivMLmods/Face-Age-10K"
SPLIT_CSV = Path("Face_Age_10K_4Groups_split_indices.csv")
SEED = 418
BATCH_SIZE = 64
EPOCHS = 10
LABEL_NAMES = ["age 01-20", "age 21-40", "age 41-65", "age 66+"]
LEARNING_RATE = 1e-3

# Fix common random sources so runs are easier to compare.
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Split file found: {SPLIT_CSV.exists()} ({SPLIT_CSV.resolve()})")

## 2. Load the dataset and saved split assignments

Place `Face_Age_10K_4Groups_split_indices.csv` in the same folder where Jupyter is running this notebook, or change `SPLIT_CSV` above to its full path. Each row in the file records the original dataset row, its age label, and its partition. We verify the expected sizes and that row indices do not overlap.


In [ ]:
if not SPLIT_CSV.exists():
    raise FileNotFoundError(
        f"Could not find {SPLIT_CSV.resolve()}. Put the CSV beside the notebook or update SPLIT_CSV."
    )

split_table = pd.read_csv(SPLIT_CSV)
source_data = load_dataset(DATASET_ID)["train"]
label_names = LABEL_NAMES
num_classes = len(label_names)

expected_splits = {"train", "validation", "test"}
if set(split_table["split"].unique()) != expected_splits:
    raise ValueError(f"Expected partitions {expected_splits}; found {set(split_table['split'].unique())}")
if split_table["row_index"].duplicated().any():
    raise ValueError("The split file contains duplicate row indices.")
if split_table["row_index"].min() < 0 or split_table["row_index"].max() >= len(source_data):
    raise ValueError("The split file contains a row index outside the loaded dataset.")

print(f"Dataset rows: {len(source_data):,}")
print("Label order (output neuron 0 through 3):")
for i, label in enumerate(label_names):
    print(f"  {i}: {label}")
print("\nPartition sizes:")
print(split_table["split"].value_counts().reindex(["train", "validation", "test"]))
print("\nClass counts per partition:")
display(pd.crosstab(split_table["label"], split_table["split"]).reindex(label_names))

## 3. Prepare image batches

PyTorch's `Dataset` object below fetches one image at a time by its original row index. The transform converts each image to RGB, resizes it to 200×200, and changes pixel values into a tensor scaled from 0 to 1. Data loaders group examples into batches, which is more efficient than processing one image per update.


In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),  # converts RGB pixels to [channels, height, width] in [0, 1]
])

class FaceAgeDataset(Dataset):
    def __init__(self, source, rows, label_names, transform):
        self.source = source
        self.rows = rows.reset_index(drop=True)
        self.label_to_id = {name: i for i, name in enumerate(label_names)}
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, position):
        row_info = self.rows.iloc[position]
        sample = self.source[int(row_info["row_index"])]
        image = sample["image"].convert("RGB")
        image_tensor = self.transform(image)
        label_id = self.label_to_id[row_info["label"]]
        return image_tensor, torch.tensor(label_id, dtype=torch.long)

split_rows = {name: split_table.loc[split_table["split"] == name].copy()
              for name in ["train", "validation", "test"]}
datasets = {name: FaceAgeDataset(source_data, rows, label_names, image_transform)
            for name, rows in split_rows.items()}

# num_workers=0 is the most reliable setting for beginner Windows setups.
loaders = {
    name: DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=(name == "train"),
                     num_workers=0, pin_memory=(device.type == "cuda"))
    for name, dataset in datasets.items()
}

images, labels = next(iter(loaders["train"]))
print(f"One batch has image tensor shape {tuple(images.shape)}")
print(f"Pixel range in this batch: {images.min().item():.3f} to {images.max().item():.3f}")
print(f"Label tensor shape: {tuple(labels.shape)}")

## 4. Define a small CNN

A convolution layer learns local visual patterns, such as edges and textures. Max pooling reduces image size between blocks. The final classifier converts the learned features into four-class scores. Dropout is a regularization step that helps reduce overfitting. This is a simple baseline architecture, not a claim that it is optimal.


In [ ]:
class SmallAgeCNN(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 200 -> 100 pixels per side

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 100 -> 50

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 50 -> 25

            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.30),
            nn.Linear(128, number_of_classes),
        )

    def forward(self, images):
        return self.classifier(self.features(images))

model = SmallAgeCNN(num_classes).to(device)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 5. Set class weights and training helpers

The four merged groups still have unequal counts; the oldest combined group has fewer examples than the youngest group. We compute a class weight from the **training partition only**: rarer classes receive higher weight in the loss. The validation and test data do not influence these weights.


In [ ]:
train_label_ids = split_rows["train"]["label"].map({name: i for i, name in enumerate(label_names)}).to_numpy()
train_counts = np.bincount(train_label_ids, minlength=num_classes)
class_weights = len(train_label_ids) / (num_classes * train_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)

print(pd.DataFrame({"age group": label_names,
                    "training examples": train_counts,
                    "loss weight": class_weights.round(3)}))

loss_function = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    true_ids, predicted_ids = [], []
    total_loss, total_examples = 0.0, 0
    for batch_images, batch_labels in loader:
        batch_images = batch_images.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)
        logits = model(batch_images)
        loss = loss_function(logits, batch_labels)
        total_loss += loss.item() * batch_labels.size(0)
        total_examples += batch_labels.size(0)
        true_ids.extend(batch_labels.cpu().numpy())
        predicted_ids.extend(logits.argmax(dim=1).cpu().numpy())
    return {
        "loss": total_loss / total_examples,
        "accuracy": accuracy_score(true_ids, predicted_ids),
        "macro_f1": f1_score(true_ids, predicted_ids, average="macro", zero_division=0),
        "true_ids": np.asarray(true_ids),
        "predicted_ids": np.asarray(predicted_ids),
    }

## 6. Train and choose the best validation checkpoint

Run this cell to train. Each epoch loops over training batches, updates model weights, and then measures validation performance. The best model is selected by validation macro-F1 and saved to a file. Training prints progress; it does not use the test set.


In [ ]:
best_validation_macro_f1 = -1.0
best_model_path = Path("face_age_cnn_best.pt")
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, seen = 0.0, 0

    for batch_images, batch_labels in loaders["train"]:
        batch_images = batch_images.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_images)
        loss = loss_function(logits, batch_labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch_labels.size(0)
        seen += batch_labels.size(0)

    train_loss = running_loss / seen
    validation_metrics = evaluate(model, loaders["validation"])
    history.append({"epoch": epoch, "train_loss": train_loss,
                    "validation_loss": validation_metrics["loss"],
                    "validation_accuracy": validation_metrics["accuracy"],
                    "validation_macro_f1": validation_metrics["macro_f1"]})

    if validation_metrics["macro_f1"] > best_validation_macro_f1:
        best_validation_macro_f1 = validation_metrics["macro_f1"]
        torch.save({"model_state_dict": model.state_dict(),
                    "label_names": label_names,
                    "seed": SEED,
                    "validation_macro_f1": best_validation_macro_f1}, best_model_path)

    print(f"Epoch {epoch:02d}/{EPOCHS} | train loss {train_loss:.4f} | "
          f"validation loss {validation_metrics['loss']:.4f} | "
          f"validation accuracy {validation_metrics['accuracy']:.3f} | "
          f"validation macro-F1 {validation_metrics['macro_f1']:.3f}")

history = pd.DataFrame(history)
print(f"\nBest validation macro-F1: {best_validation_macro_f1:.3f}")
print(f"Best checkpoint saved to: {best_model_path.resolve()}")

## 7. Review learning curves

A training loss that keeps falling while validation loss rises can indicate overfitting. The best checkpoint is chosen by validation macro-F1, but the curves still help us understand how learning progressed.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["epoch"], history["train_loss"], marker="o", label="train")
axes[0].plot(history["epoch"], history["validation_loss"], marker="o", label="validation")
axes[0].set(title="Loss by epoch", xlabel="Epoch", ylabel="Weighted cross-entropy loss")
axes[0].legend()

axes[1].plot(history["epoch"], history["validation_accuracy"], marker="o", label="accuracy")
axes[1].plot(history["epoch"], history["validation_macro_f1"], marker="o", label="macro-F1")
axes[1].set(title="Validation scores by epoch", xlabel="Epoch", ylabel="Score", ylim=(0, 1))
axes[1].legend()
plt.tight_layout()
plt.show()

## 8. Final test evaluation

After training and checkpoint selection are complete, evaluate the chosen model on the test partition. This is the final held-out result for this baseline. If you later change architecture or training settings, use validation results to make those choices and keep the same test set for a fair comparison.


In [ ]:
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
test_metrics = evaluate(model, loaders["test"])

print(f"Test accuracy: {test_metrics['accuracy']:.3f}")
print(f"Test macro-F1: {test_metrics['macro_f1']:.3f}")
print("\nPer-class results:")
print(classification_report(test_metrics["true_ids"], test_metrics["predicted_ids"],
                            labels=list(range(num_classes)), target_names=label_names,
                            zero_division=0, digits=3))

ConfusionMatrixDisplay.from_predictions(
    test_metrics["true_ids"], test_metrics["predicted_ids"],
    labels=list(range(num_classes)), display_labels=label_names,
    xticks_rotation=45, cmap="Blues", values_format="d"
)
plt.title("Held-out test confusion matrix")
plt.tight_layout()
plt.show()

## 9. Interpretation and limitations

- Compare per-class precision, recall, and F1. A strong overall accuracy can hide poor performance on a smaller class.
- Confusion between adjacent age groups may reflect fuzzy group boundaries, uncertain labels, or visual similarity; it is not automatically a code defect.
- This dataset is tagged `Synthetic` by its repository, but its card does not explain the provenance of each image or how age labels were produced. Repository licensing does not by itself settle rights to underlying images.
- Age inference from faces is sensitive and can be inaccurate across populations and image conditions. Treat this as a classroom benchmark only; do not use it for access, eligibility, surveillance, or decisions about real people.
- Record software versions, random seed, split file, model settings, and results in the project repository. Follow the instructor's project rubric and cite AI assistance as required by the course.

No result from this baseline establishes that the data are representative, that the labels are correct, or that the model is suitable for deployment.


### Four-group trial results

The trained four-class CNN achieved **53.2% test accuracy** and **0.481 test macro-F1**. Per-class recalls were 63.3% (01-20), 63.4% (21-40), 34.0% (41-65), and 28.0% (66+). The 66+ group remains the hardest group in this baseline.

As a same-test-row reference, summing adjacent softmax probabilities from the previous eight-class checkpoint gave 50.7% accuracy and 0.452 macro-F1. The four-class model was trained directly on the merged labels; this comparison is only a baseline comparison on the fixed split. The test set has now been used for both reported comparisons, so use validation results for any further model choices.
